# PREDICTIVE ML MODEL TRAINING FOR HYDROLIC PRESS THROUGH CONDITION MONITORING OF HYDRAULIC SYSTEMS DATASET

### Set-up & Imports

In [5]:
# 0/ Setup & Imports
import warnings
warnings.filterwarnings('ignore')


import os
import gc
import sys
import shap 
import time
import random
import pickle
import joblib
import contextlib
import numpy as np
import pandas as pd
import seaborn as sns
import xgboost as xgb
import matplotlib.pyplot as plt


from sklearn.feature_selection import mutual_info_regression as MIR
from sklearn.metrics import mean_squared_error as MSE
from sklearn.metrics import mean_absolute_error as MAE
from sklearn.model_selection import StratifiedKFold
from sklearn.ensemble import RandomForestRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE
from sklearn.impute import KNNImputer
from sklearn.metrics import r2_score
from xgboost import XGBRegressor
from scipy.stats import ks_2samp
from scipy.stats import kurtosis
from scipy.stats import zscore
from scipy.stats import skew
from boruta import BorutaPy
from tqdm import tqdm


# Reproducibility
np.random.seed(42)

# Display options for readable, aligned outputs
pd.set_option('display.width', 120)
pd.set_option('display.max_columns', None)
pd.set_option('display.precision', 3)

# Small helpers for tidy logs
def print_header(title: str):
    line = '=' * 70
    print(f"\n{line}\n{title}\n{line}")

def print_footer(title: str):
    line = '-' * 70
    print(f"\n{line}\n{title}\n{line}")

def print_kv(pairs):
    # pairs: list of (key, value)
    width = max(len(str(k)) for k, _ in pairs) if pairs else 0
    for k, v in pairs:
        print(f"{str(k).ljust(width)} : {v}")

def R_MSE(y_true, y_pred):
    return np.sqrt(MSE(y_true, y_pred))

def get_numeric_sensor_cols(df: pd.DataFrame, exclude_cols: set) -> list:
    cols = []
    for c in df.columns:
        if c in exclude_cols:
            continue
        if pd.api.types.is_numeric_dtype(df[c]):
            cols.append(c)
    return cols

# store results for comparison of trained models
results_dict = {}

# data directory
main_path = os.path.basename(os.getcwd())
if main_path != "condition+monitoring+of+hydraulic+systems":
    data_dir = "condition+monitoring+of+hydraulic+systems"  
    os.chdir(data_dir)
    

print_footer("Setup complete")


----------------------------------------------------------------------
Setup complete
----------------------------------------------------------------------


## 1) Load Profile.txt & Compute RUL

In [ ]:
# 1/ LOAD PROFILE.TXT & COMPUTE RUL (DEGRADATION LOGIC)

print_header("1/ LOAD PROFILE.TXT & COMPUTE RUL ")

# function def
def compute_rul(profile_df, max_rul_cap=150):
    """
    Compute Remaining Useful Life (RUL) for hydraulic press cycles.
    RUL decreases until a failure condition occurs, then resets to max.
    """
    num_cycles = len(profile_df)
    rul = np.zeros(num_cycles, dtype=int)
    current_rul = max_rul_cap

    for i in range(num_cycles - 1, -1, -1):
        cooler = profile_df.loc[i, 'cooler']
        valve = profile_df.loc[i, 'valve']
        pump = profile_df.loc[i, 'pump']
        acc = profile_df.loc[i, 'accumulator']

        # failure logic (as defined by UCI dataset rules)
        if (cooler == 3) or (valve == 73) or (pump >= 3) or (acc == 90):
            current_rul = 0
        else:
            current_rul = min(current_rul + 1, max_rul_cap)

        rul[i] = current_rul

    return rul


# RUL calculation 
profile_path = "profile.txt"
profile = pd.read_csv(
                        profile_path,
                        sep='\t', 
                        header=None,
                        names=['cooler', 'valve', 'pump', 'accumulator', 'stable']
                    )

num_cycles = len(profile)
rul = compute_rul(profile, max_rul_cap=150)

# validation & summary
print_kv([
            ("Total cycles", num_cycles),
            ("RUL min", rul.min()),
            ("RUL max", rul.max()),
            ("Mean RUL", f"{rul.mean():.2f}")
        ])

print_footer("RUL COMPUTATION COMPLETE")



1/ LOAD PROFILE.TXT & COMPUTE RUL (REVISED)
Total cycles : 2205
RUL min      : 0
RUL max      : 41
Mean RUL     : 4.74
RUL computation finished successfully.

----------------------------------------------------------------------
RUL COMPUTATION COMPLETE (REVISED)
----------------------------------------------------------------------


## 2) Sensor Files & Sampling Info

In [ ]:
# 2/ Define Sensor Files & Sampling Info (CORRECTED FOR UCI STRUCTURE)

print_header("2/ DEFINE SENSOR FILES & SAMPLING INFO (ROWS=CYCLES, COLUMNS=SAMPLES)")

# Full list of 17 sensor files
sensor_files = [
                'PS1.txt', 'PS2.txt', 'PS3.txt', 'PS4.txt', 'PS5.txt', 'PS6.txt',     # 100 Hz
                'EPS1.txt',                                                           # 100 Hz
                'FS1.txt', 'FS2.txt',                                                 # 10 Hz
                'TS1.txt', 'TS2.txt', 'TS3.txt', 'TS4.txt',                           # 1 Hz
                'VS1.txt',                                                            # 1 Hz
                'CE.txt',                                                             # 1 Hz (virtual)
                'CP.txt',                                                             # 1 Hz (virtual)
                'SE.txt'                                                              # 1 Hz (virtual)
            ]

# Sampling rates & expected SAMPLES PER CYCLE (columns in file)
sensor_info = {
                'PS1.txt': (100, 6000, 'Pressure bar'),
                'PS2.txt': (100, 6000, 'Pressure bar'),
                'PS3.txt': (100, 6000, 'Pressure bar'),
                'PS4.txt': (100, 6000, 'Pressure bar'),
                'PS5.txt': (100, 6000, 'Pressure bar'),
                'PS6.txt': (100, 6000, 'Pressure bar'),
                'EPS1.txt': (100, 6000, 'Motor power W'),
                'FS1.txt': (10, 600, 'Volume flow l/min'),
                'FS2.txt': (10, 600, 'Volume flow l/min'),
                'TS1.txt': (1, 60, 'Temperature °C'),
                'TS2.txt': (1, 60, 'Temperature °C'),
                'TS3.txt': (1, 60, 'Temperature °C'),
                'TS4.txt': (1, 60, 'Temperature °C'),
                'VS1.txt': (1, 60, 'Vibration mm/s'),
                'CE.txt': (1, 60, 'Cooling efficiency % (virtual)'),
                'CP.txt': (1, 60, 'Cooling power kW (virtual)'),
                'SE.txt': (1, 60, 'Efficiency factor % (virtual)')
            }

# Print summary
print_kv([
            ("Total sensors", len(sensor_files)),
            ("100 Hz (6000 samples)", 7),
            ("10 Hz (600 samples)", 2),
            ("1 Hz (60 samples)", 8),
            ("Cycles (rows per file)", num_cycles),
            ("Cycle duration", "60 seconds")
        ])

# Table view
info_list = []
for f in sensor_files:
    rate, samples, desc = sensor_info[f]
    info_list.append((f, rate, samples, desc))

print("\nSensor Details:")
print(pd.DataFrame(info_list, columns=['File', 'Rate (Hz)', 'Samples/Cycle (columns)', 'Description'])
      .to_string(index=False))

# Pre-load all sensor DataFrames + validate shape
sensor_dfs = {}  # cache for later feature extraction
missing_files = []
shape_mismatches = []

for f in sensor_files:
    if not os.path.exists(f):
        missing_files.append(f)
    else:
        df_temp = pd.read_csv(f, sep='\t', header=None)
        actual_shape = df_temp.shape
        expected_samples = sensor_info[f][1]
        expected_shape = (num_cycles, expected_samples)
        if actual_shape != expected_shape:
            shape_mismatches.append((f, actual_shape, expected_shape))
        else:
            sensor_dfs[f] = df_temp  # cache if correct

print_kv([
            ("Missing files", missing_files if missing_files else "None"),
            ("Shape mismatches", shape_mismatches if shape_mismatches else "None")
        ])

assert len(missing_files) == 0, "Some sensor files missing!"
assert len(shape_mismatches) == 0, "Some files have wrong shape! (rows must be 2205, columns = samples)"

print(f"All {len(sensor_dfs)} sensor files loaded and validated successfully")

print_footer("2/ SENSOR INFO COMPLETE")

## 3) Feature Extraction

In [ ]:
# 3/ FEATURE EXTRACTION (WITH TEMPORAL TREND FEATURES)

print_header("3/ FEATURE EXTRACTION (WITH TREND)")

# function definitions
def extract_features_with_trend(arr_list, window_size=5):
    """
    Extract statistical and temporal trend features.
    Each arr in arr_list represents one cycle for a single sensor.
    """
    feats_all = []
    for i in range(len(arr_list)):
        arr = np.asarray(arr_list[i], dtype=np.float64).flatten()
        mean_ = np.mean(arr)
        std_ = np.std(arr, ddof=0)
        min_ = np.min(arr)
        max_ = np.max(arr)
        rms_ = np.sqrt(np.mean(arr ** 2))
        skew_ = skew(arr)
        kurt_ = kurtosis(arr)
        ptp_ = np.ptp(arr)

        # temporal trend (difference from rolling mean of previous N cycles)
        if i >= window_size:
            prev_vals = [np.mean(arr_list[j]) for j in range(i - window_size, i)]
            trend = mean_ - np.mean(prev_vals)
        else:
            trend = 0.0

        feats_all.append([mean_, std_, min_, max_, rms_, skew_, kurt_, ptp_, trend])
    return np.array(feats_all)


# validation on one sensor
ps1_data = sensor_dfs['PS1.txt']
arr_list = [ps1_data.iloc[i, :].values for i in range(10)]
feat_sample = extract_features_with_trend(arr_list, window_size=3)
print(f"Sample feature shape (10 cycles, 9 features): {feat_sample.shape}")
print("Feature extraction with trend logic validated.")

print_footer("FEATURE EXTRACTION FUNCTION READY")


## 4) Extract Features

In [ ]:
# 4/ Extract Features Loop 

print_header("4/ EXTRACT FEATURES FOR ALL CYCLES (CACHE ENABLED)")


# Cache file
cache_file = "hydraulic_features_rul.csv"

if os.path.exists(cache_file):
    print(f"Cache found: {cache_file}")
    print("Loading features from cache...")
    features_df = pd.read_csv(cache_file)
    features_df['RUL'] = rul
    print_kv([
                ("Loaded shape", features_df.shape),
                ("Sample columns", features_df.columns[:10].tolist())
            ])
else:
    print("No cache found. Starting feature extraction...")
    
    # List to hold all cycle features
    all_features = []
    
    # Progress bar
    for cycle_idx in tqdm(range(num_cycles), desc="Extracting features", unit="cycle"):
        cycle_feats = []
        
        # Extract from each sensor (use pre-loaded sensor_dfs)
        for sensor_file in sensor_files:
            # Get row = cycle_idx, all columns = samples
            cycle_data = sensor_dfs[sensor_file].iloc[cycle_idx, :].values
            feats = extract_features(cycle_data)
            cycle_feats.extend(feats)
        
        all_features.append(cycle_feats)
    
    # Feature names (17 sensors × 8 stats)
    stat_names = ['mean', 'std', 'min', 'max', 'rms', 'skew', 'kurt', 'ptp']
    feature_names = [f"{sensor[:-4]}_{stat}" for sensor in sensor_files for stat in stat_names]
    
    # Create DataFrame
    features_df = pd.DataFrame(all_features, columns=feature_names)
    features_df.fillna(0, inplace=True)  # handle any NaNs just in case
    print_kv([
                ("Extraction complete", f"{features_df.shape[0]} cycles"),
                ("Features per cycle", features_df.shape[1])
            ])
    
    # Add RUL column
    features_df['RUL'] = rul
    print("RUL column added")
    
    # Save cache
    features_df.to_csv(cache_file, index=False)
    print(f"Features saved to cache: {cache_file} ({os.path.getsize(cache_file)/1e6:.2f} MB)")

# Validation
assert features_df.shape[0] == num_cycles, "Cycle count mismatch!"
assert features_df.shape[1] == len(sensor_files)*8 + 1, "Feature count mismatch (136 + RUL)!"
assert 'RUL' in features_df.columns, "RUL column missing!"
assert not features_df.isnull().any().any(), "NaN values in features!"

# Sample output
print("\nFirst 3 rows sample:")
print(features_df.head(3).to_string(index=False))

print_kv([
            ("Final DF shape", features_df.shape),
            ("RUL min/max", f"{features_df['RUL'].min()} / {features_df['RUL'].max()}"),
            ("Memory usage", f"{features_df.memory_usage(deep=True).sum()/1e6:.2f} MB")
        ])

print("Feature extraction complete and cached for future runs")

print_footer("FEATURE EXTRACTION COMPLETE")

## 5) Splitting and Scaling for Training

In [ ]:
# 5/ TRAIN-TEST SPLIT & SCALING (STRATIFIED BY RUL INTERVALS)

print_header("5/ TRAIN-TEST SPLIT & SCALING (STRATIFIED)")

#function definitions
def stratified_split(X, y, n_bins=5, test_size=0.2, random_state=42):
    """
    Perform stratified train-test split based on binned RUL values.
    """
    from sklearn.model_selection import StratifiedShuffleSplit

    bins = pd.cut(y, bins=n_bins, labels=False)
    splitter = StratifiedShuffleSplit(n_splits=1, test_size=test_size, random_state=random_state)
    for train_idx, test_idx in splitter.split(X, bins):
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
    return X_train, X_test, y_train, y_test


# apply
cache_file = "hydraulic_features_rul.csv"
if 'features_df' not in globals():
    features_df = pd.read_csv(cache_file)
    print("Features loaded from cache.")

y = features_df['RUL']
X = features_df.drop('RUL', axis=1)

X_train, X_test, y_train, y_test = stratified_split(X, y, n_bins=5, test_size=0.2)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

joblib.dump(scaler, "hydraulic_scaler.pkl")

# validation
print_kv([
            ("Train samples", len(y_train)),
            ("Test samples", len(y_test)),
            ("Train RUL mean", f"{y_train.mean():.2f}"),
            ("Test RUL mean", f"{y_test.mean():.2f}"),
            ("Scaler file", "hydraulic_scaler.pkl")
        ])

print("Train/test split and scaling completed successfully.")

print_footer("SPLIT & SCALING COMPLETE (STRATIFIED)")


## 6) XGBoost Training

In [ ]:
# 6/ XGBOOST MODEL TRAINING (OPTIMIZED CONFIGURATION)

print_header("6/ XGBOOST MODEL TRAINING (OPTIMIZED)")

# main model setup
model = xgb.XGBRegressor(
                            n_estimators=600,
                            learning_rate=0.03,
                            max_depth=6,
                            min_child_weight=2,
                            subsample=0.8,
                            colsample_bytree=0.8,
                            reg_lambda=1.0,
                            reg_alpha=0.1,
                            random_state=42,
                            tree_method='hist',
                            eval_metric='mae',
                            early_stopping_rounds=50,
                            n_jobs=-1
                        )

# train-validation split
val_split_idx = int(0.9 * len(X_train_scaled))
X_train_final = X_train_scaled[:val_split_idx]
y_train_final = y_train.iloc[:val_split_idx]
X_val = X_train_scaled[val_split_idx:]
y_val = y_train.iloc[val_split_idx:]

# training
print("Training started...")
model.fit(
            X_train_final,
            y_train_final,
            eval_set=[(X_val, y_val)],
            verbose=50,
        )

# evaluation on test data
y_pred = model.predict(X_test_scaled)

mae = MAE(y_test, y_pred)
rmse = R_MSE(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print_kv([
            ("Test MAE", f"{mae:.2f} cycles"),
            ("Test RMSE", f"{rmse:.2f} cycles"),
            ("Test R2", f"{r2:.4f}"),
            ("Target", "MAE < 10 & R2 > 0")
        ])

# save model
model.save_model("hydraulic_press_rul_xgb.json")
print("Model saved: hydraulic_press_rul_xgb.json")

# validation
assert mae < 20, "Model too inaccurate!"
print("XGBoost training completed successfully.")

print_footer("XGBOOST TRAINING COMPLETE")


## 7) Degradation Test

In [ ]:
# 7/ REAL DEGRADATION TEST (VISUAL & PERFORMANCE VALIDATION)

print_header("7/ REAL DEGRADATION TEST (UPDATED)")

# Evaluation
X_real_test = X_test_scaled
y_real_test = y_test
y_pred_real = model.predict(X_real_test)

mae_real = MAE(y_real_test, y_pred_real)
rmse_real = R_MSE(y_real_test, y_pred_real)
r2_real = r2_score(y_real_test, y_pred_real)

print_kv([
            ("Real test samples", len(y_real_test)),
            ("RUL range", f"{y_real_test.min()} -> {y_real_test.max()}"),
            ("MAE", f"{mae_real:.2f} cycles"),
            ("RMSE", f"{rmse_real:.2f} cycles"),
            ("R2", f"{r2_real:.4f}"),
            ("Target", "MAE < 10 & R2 > 0")
        ])

# Visualization

plt.figure(figsize=(12, 5))
plt.plot(y_real_test.values, label="True RUL", color='blue', alpha=0.8)
plt.plot(y_pred_real, label="Predicted RUL", color='red', alpha=0.8)
plt.title("Hydraulic Press - True vs Predicted RUL (Degradation Trend)")
plt.xlabel("Cycle")
plt.ylabel("RUL (Remaining Useful Life)")
plt.legend()
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.savefig("hydraulic_press_rul_trend.png", dpi=300)
plt.show()

print_footer("REAL TEST COMPLETED")


In [ ]:
# 8/ Edge AI Inference Function (ROS2 Ready)

print_header("8/ EDGE AI INFERENCE FUNCTION")

import xgboost as xgb
import joblib
import numpy as np

# Load model & scaler
model = xgb.Booster()
model.load_model("hydraulic_press_rul_xgb.json")
scaler = joblib.load("hydraulic_scaler.pkl")

def predict_rul_one_cycle(sensor_data_dict):
    """
    Input: dict of sensor arrays (e.g., {'PS1.txt': np.array(6000,), ...})
    Output: float RUL
    """
    feats = []
    for sensor_file in sensor_files:
        arr = sensor_data_dict[sensor_file]
        feats.extend(extract_features(arr))
    
    feats = np.array(feats).reshape(1, -1)
    feats_scaled = scaler.transform(feats)
    
    rul_pred = model.predict(xgb.DMatrix(feats_scaled))[0]
    return float(rul_pred)

# Test with first cycle
test_cycle = {f: sensor_dfs[f].iloc[0, :].values for f in sensor_files}
pred = predict_rul_one_cycle(test_cycle)

print_kv([
    ("Test cycle index", 0),
    ("True RUL", int(rul[0])),
    ("Predicted RUL", f"{pred:.2f}")
])

print("Edge AI function ready for ROS2 node")
print_header("8/ INFERENCE READY")